In [1]:
from pathlib import Path



In [36]:

def extxyz_to_h5(path, dtype=np.float32):
    path = Path(path)
    out = path.with_suffix(".h5")
    frames = read(path, index=":")
    first = frames[0]
    opts = dict(compression="gzip", shuffle=True)

    with h5py.File(out, "w") as f:
        f.create_dataset("positions", data=np.array([a.positions for a in frames], dtype=dtype), **opts)
        f.create_dataset("cell", data=np.array([a.cell.array for a in frames], dtype=dtype), **opts)
        f.create_dataset("atomic_numbers", data=first.numbers)
        f.create_dataset("pbc", data=first.pbc)

        # per-atom arrays, e.g. momenta, tags
        for key in first.arrays:
            if key in ("positions", "numbers"):
                continue
            data = np.array([a.arrays[key] for a in frames])
            if data.dtype.kind == "f":
                data = data.astype(dtype)
            f.create_dataset(f"arrays/{key}", data=data, **opts)

        # per-frame info, e.g. temperature_K
        for key in first.info:
            data = np.array([a.info[key] for a in frames])
            if data.dtype.kind in "OU":
                print("skipped (not numeric):", key)
                continue
            f.create_dataset(f"info/{key}", data=data, **opts)

        # energy stays as is, per-atom results like forces use dtype
        if first.calc is not None:
            for key in first.calc.results:
                data = np.array([a.calc.results[key] for a in frames])
                if data.ndim == 3 and data.dtype.kind == "f":
                    data = data.astype(dtype)
                f.create_dataset(f"calc/{key}", data=data, **opts)
    return out

In [ ]:
ROOT = Path(".")
KEEP = {"input.xyz", "final.traj", "log.out"}
DRY_RUN = True  # set to False to actually delete

for d in ROOT.iterdir():
    if not d.is_dir():
        continue
    for f in d.iterdir():
        if f.is_file() and f.name not in KEEP:
            print("delete:", f)
            if not DRY_RUN:
                f.unlink()

In [51]:
import gzip
import shutil
from pathlib import Path

ROOT = Path("./PARTICLE_SIZE/new_main")
DELETE = ["*.py", "*.extxyz", "*.txt", "*.sbatch", "slurm*"]
GZIP = []
DRY_RUN = False  # set to False to actually do it

for d in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    for pattern in DELETE:
        for f in d.glob(pattern):
            if f.is_file():
                print("delete:", f)
                if not DRY_RUN:
                    f.unlink()

    for name in GZIP:
        f = d / name
        if f.exists():
            print("gzip:  ", f)
            if not DRY_RUN:
                with open(f, "rb") as src, gzip.open(f.with_name(f.name + ".gz"), "wb") as dst:
                    shutil.copyfileobj(src, dst)
                f.unlink()

delete: PARTICLE_SIZE/new_main/x_1500_1000/slurm-452272.out
delete: PARTICLE_SIZE/new_main/x_1500_1100/slurm-456417.out
delete: PARTICLE_SIZE/new_main/x_1500_1100/slurm-456425.out
delete: PARTICLE_SIZE/new_main/x_1500_1200/slurm-456418.out
delete: PARTICLE_SIZE/new_main/x_1500_1200/slurm-456426.out
delete: PARTICLE_SIZE/new_main/x_1500_1300/slurm-466867.out
delete: PARTICLE_SIZE/new_main/x_1500_600/slurm-456421.out
delete: PARTICLE_SIZE/new_main/x_1500_600/slurm-456413.out
delete: PARTICLE_SIZE/new_main/x_1500_700/slurm-456414.out
delete: PARTICLE_SIZE/new_main/x_1500_700/slurm-456422.out
delete: PARTICLE_SIZE/new_main/x_1500_800/slurm-456415.out
delete: PARTICLE_SIZE/new_main/x_1500_800/slurm-456423.out
delete: PARTICLE_SIZE/new_main/x_1500_900/slurm-456424.out
delete: PARTICLE_SIZE/new_main/x_1500_900/slurm-456416.out
delete: PARTICLE_SIZE/new_main/x_3000_1000/slurm-453589.out
delete: PARTICLE_SIZE/new_main/x_3000_1000/slurm-453591.out
delete: PARTICLE_SIZE/new_main/x_3000_1000/slurm

In [53]:
# from helios.settings.edit_h5 import h5_to_extxyz
from mylib.torchsim.utils import h5_to_extxyz

h5_to_extxyz("./PARTICLE_SIZE/new_main/x_1500_1000/12.h5")

PosixPath('PARTICLE_SIZE/new_main/x_1500_1000/12.extxyz')

In [37]:
extxyz_to_h5("./helios/cool_down_b/500_12_0_1300_1/full_trajectory.extxyz")

PosixPath('helios/cool_down_b/500_12_0_1300_1/full_trajectory.h5')

In [46]:
from pathlib import Path

ROOT = Path("./CuGa-initial/inputs").expanduser()
DELETE = ["*_T[0-9]*.h5", "main.py","*.sbatch", "*.txt", "DONE"]
DRY_RUN = True  # set to False to actually do it

for d in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    src = d / "*.extxyz"
    if not src.exists():
        print("skip, no full_trajectory.extxyz:", d)
        continue

    print("convert:", src)
    if not DRY_RUN:
        extxyz_to_h5(src)

    for pattern in DELETE:
        for f in d.glob(pattern):
            print("delete: ", f)
            if not DRY_RUN:
                f.unlink()

skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_1000
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_1100
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_1200
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_1300
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_600
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_700
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_800
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_1500_900
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_1000
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_1100
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_1200
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_1300
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_600
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_700
skip, no full_trajectory.extxyz: CuGa-initial/inputs/x_3000_800
skip, no full_trajectory.extxyz:

In [44]:
from pathlib import Path

ROOT = Path("~/PHD/1-CuGa/helios/sim").expanduser()
DELETE = ["new.py","full_traj*.extxyz", "*.txt", "DONE", "musica.sbatch"]
NEW_NAME = "fulltraj.h5"
DRY_RUN = False  # set to False to actually do it

for d in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    main = d / f"{d.name}.h5"
    target = d / NEW_NAME

    if main.exists():
        print("rename:", main, "->", target)
        if not DRY_RUN:
            main.rename(target)
    elif not target.exists():
        print("skip, no h5:", d)
        continue

    for pattern in DELETE:
        for f in d.glob(pattern):
            print("delete:", f)
            if not DRY_RUN:
                f.unlink()

delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1000_1/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1000_2/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1000_3/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1000_4/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1000_5/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1050_1/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1050_2/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1050_3/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1050_4/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1050_5/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1100_1/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1100_2/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1100_3/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1100_4/new.py
delete: /home/chris/PHD/1-CuGa/helios/sim/1500_12_0_1100_5/new.py
delete: /h

In [35]:
import h5py

def show(path):
    print(path)
    with h5py.File(path, "r") as f:
        f.visititems(lambda n, o: print("  ", n, o.shape, o.dtype, o.compression, o.shuffle)
                     if isinstance(o, h5py.Dataset) else None)

show("./helios/cool_down_b/500_12_0_1300_1/full_trajectory.h5")
show("./helios/cool_down_b/500_12_0_1300_1/500_12_0_1300_1_T1000.h5")

./helios/cool_down_b/500_12_0_1300_1/full_trajectory.h5
   arrays/momenta (867, 459, 3) float64 gzip False
   atomic_numbers (459,) int64 None False
   calc/energy (867,) float64 gzip False
   calc/forces (867, 459, 3) float64 gzip False
   cell (867, 3, 3) float32 None False
   info/temperature_K (867,) int64 None False
   pbc (3,) bool None False
   positions (867, 459, 3) float32 gzip False
./helios/cool_down_b/500_12_0_1300_1/500_12_0_1300_1_T1000.h5
   data/atomic_numbers (1, 459) int32 gzip True
   data/cell (51, 3, 3) float32 gzip True
   data/energy (51, 1) float32 gzip True
   data/forces (51, 459, 3) float32 gzip True
   data/masses (1, 459) float32 gzip True
   data/pbc (3,) uint8 gzip True
   data/positions (51, 459, 3) float32 gzip True
   data/velocities (51, 459, 3) float32 gzip True
   steps/atomic_numbers (1,) int32 gzip True
   steps/cell (51,) int32 gzip True
   steps/energy (51,) int32 gzip True
   steps/forces (51,) int32 gzip True
   steps/masses (1,) int32 gzip T

In [ ]:
from pathlib import Path


def gzip_file(path) -> Path:
    path = Path(path)
    gz = path.with_name(path.name + ".gz")
    with open(path, "rb") as src, gzip.open(gz, "wb") as dst:
        shutil.copyfileobj(src, dst)
    path.unlink()
    return gz

In [24]:
import h5py
import numpy as np
from pathlib import Path
from ase.io import read, write
from ase.calculators.singlepoint import SinglePointCalculator
from ase import Atoms

def extxyz_to_h5(path):
    path = Path(path)
    out = path.with_suffix(".h5")
    frames = read(path, index=":")
    first = frames[0]

    with h5py.File(out, "w") as f:
        f.create_dataset("positions", data=np.array([a.positions for a in frames], dtype=np.float32), compression="gzip")
        f.create_dataset("cell", data=np.array([a.cell.array for a in frames], dtype=np.float32))
        f.create_dataset("atomic_numbers", data=first.numbers)
        f.create_dataset("pbc", data=first.pbc)

        # per-atom arrays, e.g. velocities, momenta, tags
        for key in first.arrays:
            if key in ("positions", "numbers"):
                continue
            f.create_dataset(f"arrays/{key}", data=np.array([a.arrays[key] for a in frames]), compression="gzip")

        # per-frame info, e.g. step, time, temperature
        for key in first.info:
            data = np.array([a.info[key] for a in frames])
            if data.dtype.kind in "OU":
                print("skipped (not numeric):", key)
                continue
            f.create_dataset(f"info/{key}", data=data)

        # energy, forces, stress, if present
        if first.calc is not None:
            for key in first.calc.results:
                f.create_dataset(f"calc/{key}", data=np.array([a.calc.results[key] for a in frames]), compression="gzip")
    return out


def h5_to_extxyz(path):
    path = Path(path)
    out = path.with_suffix(".extxyz")

    with h5py.File(path, "r") as f:
        positions = f["positions"][:]
        cell = f["cell"][:]
        numbers = f["atomic_numbers"][:]
        pbc = f["pbc"][:].astype(bool)
        arrays = {k: d[:] for k, d in f["arrays"].items()} if "arrays" in f else {}
        info = {k: d[:] for k, d in f["info"].items()} if "info" in f else {}
        calc = {k: d[:] for k, d in f["calc"].items()} if "calc" in f else {}

    frames = []
    for i in range(len(positions)):
        atoms = Atoms(numbers=numbers, positions=positions[i], cell=cell[i], pbc=pbc)
        for k, v in arrays.items():
            atoms.arrays[k] = v[i]
        for k, v in info.items():
            atoms.info[k] = v[i].item() if np.ndim(v[i]) == 0 else v[i]
        if calc:
            atoms.calc = SinglePointCalculator(atoms, **{k: v[i] for k, v in calc.items()})
        frames.append(atoms)

    write(out, frames, format="extxyz")
    return out

In [17]:
extxyz_to_h5("./cuga_gen_f/MD_short_u/12_100.extxyz")

PosixPath('cuga_gen_f/MD_short_u/12_100.h5')

In [26]:

def same_extxyz(path_a, path_b, atol=1e-5, rtol=1e-6):
    A = read(path_a, ":")
    B = read(path_b, ":")
    if len(A) != len(B):
        return False

    def close(x, y):
        x, y = np.asarray(x), np.asarray(y)
        if x.shape != y.shape:
            return False
        if x.dtype.kind in "iuf" and y.dtype.kind in "iuf":
            return np.allclose(x, y, rtol=rtol, atol=atol)
        return np.array_equal(x, y)

    for a, b in zip(A, B):
        if not (np.array_equal(a.numbers, b.numbers)
                and close(a.cell.array, b.cell.array)
                and np.array_equal(a.pbc, b.pbc)):
            return False
        if a.arrays.keys() != b.arrays.keys() or not all(close(a.arrays[k], b.arrays[k]) for k in a.arrays):
            return False
        if a.info.keys() != b.info.keys() or not all(close(a.info[k], b.info[k]) for k in a.info):
            return False
        ra = a.calc.results if a.calc else {}
        rb = b.calc.results if b.calc else {}
        if ra.keys() != rb.keys() or not all(close(ra[k], rb[k]) for k in ra):
            return False
    return True

In [18]:
base = Path("./cuga_gen_f/MD_short_u")
for file in base.iterdir():
    try: 
        extxyz_to_h5(file)
    except Exception as e:
        print(e)
        pass

In [30]:
h5_to_extxyz(base/"2_100.h5")

PosixPath('cuga_gen_f/MD_short_u/2_100.extxyz')

In [31]:
same_extxyz(base/"testc.extxyz", base/"2_100.extxyz")

True

In [7]:
gzip_file("./PARTICLE_SIZE/new_main/x_1500_1000/12.extxyz")

PosixPath('PARTICLE_SIZE/new_main/x_1500_1000/12.extxyz.gz')

In [8]:
from pathlib import Path
from ase.io import iread
from ase.io.netcdftrajectory import NetCDFTrajectory

def extxyz_to_nc(path, double=False):
    path = Path(path)
    out = path.with_suffix(".nc")
    traj = NetCDFTrajectory(str(out), mode="w", double=double)
    for atoms in iread(path, index=":"):
        traj.write(atoms)
    traj.close()
    return out

In [10]:
extxyz_to_nc("./PARTICLE_SIZE/new_main/x_1500_1000/15.extxyz")

PosixPath('PARTICLE_SIZE/new_main/x_1500_1000/15.nc')

In [11]:
import h5py

In [14]:
import h5py

with h5py.File("./PARTICLE_SIZE/new_main/x_1500_1000/12.h5", "r") as f:
    f.visititems(lambda name, obj: print(name, obj))
    print(dict(f.attrs))

data <HDF5 group "/data" (5 members)>
data/atomic_numbers <HDF5 dataset "atomic_numbers": shape (1, 1289), type "<i4">
data/cell <HDF5 dataset "cell": shape (1001, 3, 3), type "<f4">
data/masses <HDF5 dataset "masses": shape (1, 1289), type "<f4">
data/pbc <HDF5 dataset "pbc": shape (3,), type "|u1">
data/positions <HDF5 dataset "positions": shape (1001, 1289, 3), type "<f4">
header <HDF5 group "/header" (0 members)>
metadata <HDF5 group "/metadata" (0 members)>
steps <HDF5 group "/steps" (5 members)>
steps/atomic_numbers <HDF5 dataset "atomic_numbers": shape (1,), type "<i4">
steps/cell <HDF5 dataset "cell": shape (1001,), type "<i4">
steps/masses <HDF5 dataset "masses": shape (1,), type "<i4">
steps/pbc <HDF5 dataset "pbc": shape (1,), type "<i4">
steps/positions <HDF5 dataset "positions": shape (1001,), type "<i4">
{'CLASS': np.bytes_(b'GROUP'), 'FILTERS': np.int64(65793), 'PYTABLES_FORMAT_VERSION': np.bytes_(b'2.1'), 'TITLE': Empty(dtype=dtype('S1')), 'VERSION': np.bytes_(b'1.0')}
